# 🧪 06 - Describing Data Lab: Palmer Penguins

A research team has measured 344 penguins on three islands in Antarctica and sent you `penguins.csv`. They want a **numeric description** of the data before anyone draws a single chart:

- What's in the file, and what does each column mean?
- What's typical for each measurement, and how much does it vary?
- Are there outliers, and are they mistakes?
- How do the three species differ?
- Which measurements are related, and does that hold up within each species?

You'll answer with everything from notebooks 01–05, and save a small report. In **Module 08's lab**, you'll turn these same findings into figures, so keep your answers. You'll check them against the charts.

## ✅ Learning Goals
- Run a full descriptive workflow, from raw file to saved summary tables
- Choose statistics that match each column's type and shape
- Separate real outliers from errors, overall and within groups
- Compare groups fairly, and test whether a relationship survives splitting by group

```text
Load → Inspect → Clean (Module 06) → Data dictionary (01)
     → Center & spread (02) → Shape & outliers (03)
     → Compare groups (04) → Relationships (05) → Export the report
```

## 0. Setup
Everything this lab saves goes into **`06-OutputFiles`**. Upload that folder along with this notebook.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path('06-OutputFiles')
OUT.mkdir(exist_ok=True)

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

## 1. Load and Look (Change Nothing Yet)

In [2]:
penguins = pd.read_csv('penguins.csv')
print(penguins.shape)
penguins.head()

(344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [3]:
penguins.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


In [4]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

**One row = one penguin.** Four measurements are each missing twice, and `sex` is missing 11 times. Are the measurement gaps in the same two rows?

In [5]:
penguins[penguins[MEASUREMENTS].isna().any(axis=1)]

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN


Yes: two penguins with **no measurements at all**. They can't contribute anything to a description of measurements.

## 2. Clean (Module 06)
1. Drop the two rows with no measurements.
2. Keep penguins with unknown `sex`: their measurements are still useful. Just remember they'll drop out of any by-sex comparison.
3. Tidy `sex` from `MALE`/`FEMALE` to `Male`/`Female`.
4. Record meaning with dtypes (notebook 01): `species`, `island`, and `sex` are nominal → `category`.

In [6]:
clean = penguins.dropna(subset=MEASUREMENTS).copy()
clean['sex'] = clean['sex'].str.title()

for col in ['species', 'island', 'sex']:
    clean[col] = clean[col].astype('category')

print('Rows before:', len(penguins), '| after:', len(clean))
print(clean['sex'].value_counts(dropna=False))
clean.dtypes

Rows before: 344 | after: 342
sex
Male      168
Female    165
NaN         9
Name: count, dtype: int64


species              category
island               category
bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm     float64
body_mass_g           float64
sex                  category
dtype: object

## 3. Data Dictionary (Notebook 01)

In [7]:
data_dictionary = pd.DataFrame({
    'column': clean.columns,
    'meaning': ['Penguin species', 'Island where measured', 'Bill (beak) length',
                'Bill (beak) depth', 'Flipper length', 'Body mass', 'Sex'],
    'kind': ['qualitative', 'qualitative', 'quantitative', 'quantitative',
             'quantitative', 'quantitative', 'qualitative'],
    'scale': ['nominal', 'nominal', 'ratio', 'ratio', 'ratio', 'ratio', 'nominal'],
    'dtype': clean.dtypes.astype(str).values,
    'missing': clean.isna().sum().values,
    'distinct': clean.nunique().values,
})
data_dictionary.to_csv(OUT / 'penguins_data_dictionary.csv', index=False)
data_dictionary

,column,meaning,kind,scale,dtype,missing,distinct
0,species,Penguin species,qualitative,nominal,category,0,3
1,island,Island where measured,qualitative,nominal,category,0,3
2,bill_length_mm,Bill (beak) length,quantitative,ratio,float64,0,164
3,bill_depth_mm,Bill (beak) depth,quantitative,ratio,float64,0,80
4,flipper_length_mm,Flipper length,quantitative,ratio,float64,0,55
5,body_mass_g,Body mass,quantitative,ratio,float64,0,94
6,sex,Sex,qualitative,nominal,category,9,2


## 4. Center and Spread (Notebook 02)
One table with every statistic from notebooks 02 and 03, one row per measurement:

In [8]:
overall = clean[MEASUREMENTS].describe().T
overall['IQR'] = overall['75%'] - overall['25%']
overall['skew'] = clean[MEASUREMENTS].skew()
overall.round(2)

,count,mean,std,min,25%,50%,75%,max,IQR,skew
bill_length_mm,342.0,43.92,5.46,32.1,39.22,44.45,48.5,59.6,9.27,0.05
bill_depth_mm,342.0,17.15,1.97,13.1,15.60,17.30,18.7,21.5,3.10,-0.14
flipper_length_mm,342.0,200.92,14.06,172.0,190.00,197.00,213.0,231.0,23.00,0.35
body_mass_g,342.0,4201.75,801.95,2700.0,3550.00,4050.00,4750.0,6300.0,1200.00,0.47


All four measurements have a mean close to their median, and |skew| below 0.5: no strong lopsidedness like Titanic's fares. Mean and std are reasonable summaries here.

But "roughly symmetric overall" can hide something. This file mixes **three species** of very different sizes. Keep that in mind for section 6.

## 5. Shape and Outliers (Notebook 03)
Reuse the fence function from notebook 03, and count outliers in each measurement:

In [9]:
def iqr_fences(series, k=1.5):
    """Return the (lower, upper) outlier fences for a numeric Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


for col in MEASUREMENTS:
    low, high = iqr_fences(clean[col])
    n_out = ((clean[col] < low) | (clean[col] > high)).sum()
    print(f'{col:<18} fences {low:8.1f} to {high:8.1f}   outliers: {n_out}')

bill_length_mm     fences     25.3 to     62.4   outliers: 0
bill_depth_mm      fences     10.9 to     23.3   outliers: 0
flipper_length_mm  fences    155.5 to    247.5   outliers: 0
body_mass_g        fences   1750.0 to   6550.0   outliers: 0


No outliers at all across the whole dataset. Now ask notebook 03's Practice 3 question: is a value unusual **for its own species**? Compute the fences within each species with `groupby().transform()`, which lines each group's statistic up with the original rows:

In [10]:
def outliers_within(df, col, group):
    """Boolean mask: is each value outside its own group's IQR fences?"""
    q1 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.25))
    q3 = df.groupby(group, observed=True)[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)


within = pd.DataFrame({col: outliers_within(clean, col, 'species') for col in MEASUREMENTS})
within.sum()

bill_length_mm       1
bill_depth_mm        1
flipper_length_mm    2
body_mass_g          2
dtype: int64

In [11]:
clean[within.any(axis=1)].sort_values('species')

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
19,Adelie,Torgersen,46.0,21.5,194.0,4200.0,Male
28,Adelie,Biscoe,37.9,18.6,172.0,3150.0,Female
129,Adelie,Torgersen,44.1,18.0,210.0,4000.0,Male
189,Chinstrap,Dream,52.0,20.7,210.0,4800.0,Male
190,Chinstrap,Dream,46.9,16.6,192.0,2700.0,Female
253,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,Male


A few penguins are unusual **for their species**, even though nothing is unusual overall. Look at the rows: are any values impossible? (A small Chinstrap is still a possible penguin. A 30 g penguin, or a 3-meter flipper, would not be.) These look like real, unusual animals, so we **keep** them.

> 🧠 Whether a value is an outlier depends on **which group you compare it to**.

## 6. Comparing Species (Notebook 04)

In [12]:
species_summary = (
    clean
    .groupby('species', observed=True)
    .agg(
        penguins=('body_mass_g', 'size'),
        mass_mean=('body_mass_g', 'mean'),
        mass_median=('body_mass_g', 'median'),
        mass_std=('body_mass_g', 'std'),
        flipper_mean=('flipper_length_mm', 'mean'),
        bill_length_mean=('bill_length_mm', 'mean'),
        bill_depth_mean=('bill_depth_mm', 'mean'),
    )
    .round(1)
)
species_summary.to_csv(OUT / 'species_summary.csv')
species_summary

,penguins,mass_mean,mass_median,mass_std,flipper_mean,bill_length_mean,bill_depth_mean
species,,,,,,,
Adelie,151,3700.7,3700.0,458.6,190.0,38.8,18.3
Chinstrap,68,3733.1,3700.0,384.3,195.8,48.8,18.4
Gentoo,123,5076.0,5000.0,504.1,217.2,47.5,15.0


Gentoo penguins are much heavier, with longer flippers, but **shallower** bills. Hold on to that "but". It matters in section 7.

### Where Each Species Lives

In [13]:
pd.crosstab(clean['species'], clean['island'], margins=True)

island,Biscoe,Dream,Torgersen,All
species,,,,
Adelie,44,56,51,151
Chinstrap,0,68,0,68
Gentoo,123,0,0,123
All,167,124,51,342


Chinstraps were measured only on Dream, and Gentoos only on Biscoe. So "Biscoe penguins are heavy" would really be a statement about **Gentoos**: island and species are tangled together (a confounder, notebook 04). Compare islands only **within** a species that lives on more than one (Adelie).

### Species × Sex

In [14]:
mass_by_species_sex = clean.pivot_table(
    values='body_mass_g', index='species', columns='sex', aggfunc='mean', observed=True,
).round(0)
mass_by_species_sex['difference'] = mass_by_species_sex['Male'] - mass_by_species_sex['Female']
mass_by_species_sex

sex,Female,Male,difference
species,,,
Adelie,3369.0,4043.0,674.0
Chinstrap,3527.0,3939.0,412.0
Gentoo,4680.0,5485.0,805.0


Within every species, males are heavier. The penguins with unknown sex aren't in this table (section 2).

## 7. Relationships (Notebook 05)

In [15]:
corr = clean[MEASUREMENTS].corr()
corr.round(2)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
bill_length_mm,1.00,-0.24,0.66,0.60
bill_depth_mm,-0.24,1.00,-0.58,-0.47
flipper_length_mm,0.66,-0.58,1.00,0.87
body_mass_g,0.60,-0.47,0.87,1.00


In [16]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
upper.stack().dropna().rename('r').sort_values(key=abs, ascending=False).round(3)

flipper_length_mm  body_mass_g          0.871
bill_length_mm     flipper_length_mm    0.656
                   body_mass_g          0.595
bill_depth_mm      flipper_length_mm   -0.584
                   body_mass_g         -0.472
bill_length_mm     bill_depth_mm       -0.235
Name: r, dtype: float64

Flipper length and body mass are very strongly related (bigger penguins, bigger flippers). Now look at **bill length vs. bill depth**: a weak **negative** correlation. Longer bills are shallower?

Check whether that holds up **within each species**:

In [17]:
rows = []
for species, group in clean.groupby('species', observed=True):
    rows.append({
        'species': species,
        'n': len(group),
        'r_bill_length_depth': group['bill_length_mm'].corr(group['bill_depth_mm']),
    })
rows.append({'species': 'ALL (combined)', 'n': len(clean),
             'r_bill_length_depth': clean['bill_length_mm'].corr(clean['bill_depth_mm'])})

bill_corr = pd.DataFrame(rows).round(3)
bill_corr.to_csv(OUT / 'bill_correlation_by_species.csv', index=False)
bill_corr

,species,n,r_bill_length_depth
0,Adelie,151,0.391
1,Chinstrap,68,0.654
2,Gentoo,123,0.643
3,ALL (combined),342,-0.235


🤯 **Within every species, the correlation is positive, and moderate to strong.** Combined, it's negative.

This is **Simpson's paradox**: a relationship that reverses when you combine groups. The species summary in section 6 explains how. Gentoos have long bills **and** shallow bills compared with the others. Pooling a "long and shallow" group with "shorter and deeper" groups creates a downward trend that doesn't exist inside any one species. `species` is the confounder.

You found this with numbers alone, but you had to know to split by species. In Module 08's lab, one scatter plot colored by species shows it instantly, with no need to know ahead of time.

## 8. Export the Report

In [18]:
clean.to_csv(OUT / 'cleaned_penguins.csv', index=False)
overall.round(2).to_csv(OUT / 'measurement_summary.csv')
corr.round(3).to_csv(OUT / 'correlation_matrix.csv')

print('Saved:', sorted(p.name for p in OUT.glob('*.csv')))

Saved: ['bill_correlation_by_species.csv', 'cleaned_penguins.csv', 'correlation_matrix.csv', 'measurement_summary.csv', 'penguins_data_dictionary.csv', 'sex_differences_by_species.csv', 'species_summary.csv']


## ❓ Lab Questions
Answer each in the Markdown cell below, with numbers from your output:
1. What does one row represent? How many rows did you remove, and why? Why did you **keep** penguins with unknown sex?
2. Which measurement varies the most **relative to its mean**? (Hint: divide `std` by `mean`.)
3. Why were there no outliers overall, but some within species? Did you keep or remove them, and why?
4. Which species is heaviest, and by how much (mean and median) compared with the lightest?
5. Why can't you compare islands directly? Which comparison would be fair?
6. Explain the bill length vs. bill depth result in your own words, as if to someone who has never heard of Simpson's paradox.

1. One row = one penguin. Removed 2 rows with no measurements (344 → 342). Kept the 9 with unknown sex because their measurements are still valid.
2. Body mass: std/mean = 802/4202 = 0.19 (bill length 0.12, bill depth 0.12, flipper 0.07).
3. Mixing three species widens the overall fences, so nothing falls outside. Within species, 6 penguins are unusual. Kept: all values are possible for a real penguin.
4. Gentoo: mean 5076 g vs. Adelie 3701 g (+1375 g). Median 5000 g vs. 3700 g (+1300 g).
5. Chinstraps live only on Dream and Gentoos only on Biscoe, so island differences are really species differences. Fair: compare Adelies across islands.
6. Within each species, longer bills are also deeper (r = 0.39 to 0.65). Gentoos have long but shallow bills, so pooling all species makes it look like longer bills are shallower (r = −0.24).

## 🏋️ Practice

### Practice 1 — Species Spread (easy)
For each species, compute the IQR of `flipper_length_mm`. Which species is the most variable, and is it also the one with the largest std?

In [19]:
for species, group in clean.groupby('species', observed=True):
    q1 = group['flipper_length_mm'].quantile(0.25)
    q3 = group['flipper_length_mm'].quantile(0.75)
    print(f"{species:<10} IQR: {q3 - q1:5.1f}   std: {group['flipper_length_mm'].std():5.2f}")

# Chinstrap is most variable by both IQR and std.

Adelie     IQR:   9.0   std:  6.54
Chinstrap  IQR:  10.0   std:  7.13
Gentoo     IQR:   9.0   std:  6.48


### Practice 2 — A Fair Island Comparison (medium)
Filter to **Adelie** penguins only (the one species on all three islands). Compare their body mass across islands with size, mean, median, and std. Is there a real island difference among Adelies?

In [20]:
adelie = clean[clean['species'] == 'Adelie']

adelie_by_island = (
    adelie
    .groupby('island', observed=True)
    .agg(
        penguins=('body_mass_g', 'size'),
        mass_mean=('body_mass_g', 'mean'),
        mass_median=('body_mass_g', 'median'),
        mass_std=('body_mass_g', 'std'),
    )
    .round(1)
)
adelie_by_island

# No real island difference the means are 3688 to 3710 g which is smaller than the std

,penguins,mass_mean,mass_median,mass_std
island,,,,
Biscoe,44,3709.7,3750.0,487.7
Dream,56,3688.4,3575.0,455.1
Torgersen,51,3706.4,3700.0,445.1


### Practice 3 — Hunt for Another Reversal (harder)
Loop over **every pair** of measurements. For each pair, compute the combined correlation and the three within-species correlations. Flag any pair where the combined sign differs from **all three** species' signs. Is bill length/depth the only one?

In [21]:
pairs = [
    ('bill_length_mm', 'bill_depth_mm'),
    ('bill_length_mm', 'flipper_length_mm'),
    ('bill_length_mm', 'body_mass_g'),
    ('bill_depth_mm', 'flipper_length_mm'),
    ('bill_depth_mm', 'body_mass_g'),
    ('flipper_length_mm', 'body_mass_g'),
]

rows = []
for a, b in pairs:
    combined = clean[a].corr(clean[b])
    by_species = clean.groupby('species', observed=True).apply(lambda d: d[a].corr(d[b]))
    rows.append({
        'pair': f'{a} / {b}',
        'combined': combined,
        'Adelie': by_species['Adelie'],
        'Chinstrap': by_species['Chinstrap'],
        'Gentoo': by_species['Gentoo'],
        'reversal': (by_species * combined < 0).sum() == 3,
    })

pd.DataFrame(rows).round(3)

# Not the only one, Gentoos are big with shallow bills

,pair,combined,Adelie,Chinstrap,Gentoo,reversal
0,bill_length_mm / bill_depth_mm,-0.235,0.391,0.654,0.643,True
1,bill_length_mm / flipper_length_mm,0.656,0.326,0.472,0.661,False
2,bill_length_mm / body_mass_g,0.595,0.549,0.514,0.669,False
3,bill_depth_mm / flipper_length_mm,-0.584,0.308,0.580,0.707,True
4,bill_depth_mm / body_mass_g,-0.472,0.576,0.604,0.719,True
5,flipper_length_mm / body_mass_g,0.871,0.468,0.642,0.703,False


## ✏️ Challenge Task
Build a **sex comparison report**: for each species, the number of males and females, and the male − female difference in mean `body_mass_g`, `flipper_length_mm`, and `bill_depth_mm`. Save it as `OUT / 'sex_differences_by_species.csv'`.

In [22]:
sex_report = pd.crosstab(clean['species'], clean['sex']).rename(
    columns={'Female': 'n_female', 'Male': 'n_male'}
)

for col in ['body_mass_g', 'flipper_length_mm', 'bill_depth_mm']:
    table = clean.pivot_table(values=col, index='species', columns='sex', aggfunc='mean', observed=True)
    sex_report[f'{col}_diff'] = (table['Male'] - table['Female']).round(1)

sex_report.to_csv(OUT / 'sex_differences_by_species.csv')
sex_report

sex,n_female,n_male,body_mass_g_diff,flipper_length_mm_diff,bill_depth_mm_diff
species,,,,,
Adelie,73,73,674.7,4.6,1.5
Chinstrap,34,34,411.8,8.2,1.7
Gentoo,58,61,805.1,8.8,1.5


## 🔥 Challenge (Optional) — Same Workflow, New Data
Run sections 1–7 on `pokemon.csv` in a compact form: data dictionary, measurement summary with skew, outliers overall and within `Legendary`, a group comparison by `Generation`, and the strongest correlations. Save at least one table to `OUT` with a `pokemon_` prefix. Write three findings in a Markdown cell.

In [23]:
# Your code here

*Your three findings here.*

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `06-OutputFiles`. Upload that folder along with this notebook.

In [24]:
for path in sorted(OUT.rglob('*')):
    print(path)

06-OutputFiles\bill_correlation_by_species.csv
06-OutputFiles\cleaned_penguins.csv
06-OutputFiles\correlation_matrix.csv
06-OutputFiles\measurement_summary.csv
06-OutputFiles\penguins_data_dictionary.csv
06-OutputFiles\sex_differences_by_species.csv
06-OutputFiles\species_summary.csv


---
### 📝 Summary
This lab described a dataset from scratch, with numbers only:
- **Inspected** first, removed only rows with nothing to contribute, and kept partial rows
- **Recorded meaning** with a data dictionary and `category` dtypes
- **Summarized** center, spread, and shape, then checked outliers overall **and within groups**
- **Compared species** fairly, and spotted island as a stand-in for species
- **Tested a relationship within groups** and found Simpson's paradox

🎓 That's Module 07. **Module 08** picks up right here: the same questions, answered with charts. Its first notebook starts with the Anscombe datasets from notebook 05, and its lab returns to these penguins.